In [1]:
import os
os.chdir(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))

# Performance Basics

**Time: 20 min**

Compare CPU and GPU performance for the same matrix multiplication using the programs in [exercises/01-performance-basics](../exercises/01-performance-basics/).

Both programs multiply two square, double-precision matrices and accumulate into a third matrix:

```text
C = A × B + C
```

The CPU program uses Intel MKL; the GPU program uses cuBLAS. Both default to `N = 1000` and five runs.

> **Performance note**
>
> A GPU's benefit depends on the workload, problem size, precision, and what is included in the measurement. Compare the same operation and settings, repeat measurements, and distinguish computation time from total application time. A smaller printed time in this exercise is a result for this configuration, rather than a guarantee that all GPU programs are faster.

## Prepare the environment

Use a notebook kernel on an allocated Gadi GPU node. Load GCC, CUDA, and Intel MKL **before launching the notebook kernel**, so compiler headers and libraries are available to the `!` commands. The workshop module setup includes:

```bash
module load gcc/14.2.0 cuda/12.9.0 intel-mkl/2024.2.1
```

The CPU benchmark links the **sequential MKL library** (`mkl_sequential`), so this compares a GPU with a single-threaded CPU library configuration. It is not a comparison with all available CPU cores.

### Inspect the programs

In [2]:
!cat exercises/01-performance-basics/perf-gemm-cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <cublas_v2.h>
#include <cuda_runtime.h>

/**
 * Macro to check CUDA runtime errors
 */
#define CHECK_CUDA(call)                                                      \
    do {                                                                      \
        cudaError_t err = call;                                               \
        if (err != cudaSuccess) {                                             \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));             \
            exit(EXIT_FAILURE);                                               \
        }                                                                     \
    } while (0)

/**
 * Map cuBLAS status codes to human-readable strings
 */
const char* cublasGetErrorString(cublasStatus_t status) {
    switch (status) {
        case CUBLAS_STATUS_SUCCESS: return "CUBLAS_STATUS_SUCCESS";

In [3]:
!cat exercises/01-performance-basics/perf-gemm-intelmkl.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <mkl.h>

int main(void)
{
    int N = 1000;       // Matrix dimension
    int runs = 5;       // Number of GEMM runs

    // Allocate matrices on CPU
    double *A = (double *)malloc(sizeof(double) * N * N);
    double *B = (double *)malloc(sizeof(double) * N * N);
    double *C = (double *)malloc(sizeof(double) * N * N);

    if (!A || !B || !C) {
        fprintf(stderr, "Memory allocation failed\n");
        return 1;
    }

    // Initialize matrices with some values
    for (int i = 0; i < N * N; i++) {
        A[i] = 1.0;
        B[i] = 2.0;
        C[i] = 0.0;
    }

    double alpha = 1.0;
    double beta  = 1.0;

    // Time each run using clock_gettime
    struct timespec start, end;

    for (int i = 0; i < runs; i++) {
        clock_gettime(CLOCK_MONOTONIC, &start);

        // Perform GEMM: C = alpha * A * B + beta * C
        cblas_dgemm(CblasRowMajor, CblasNoTrans, CblasNoTrans,
                    N, N, N,

## Choose the benchmark settings

Change the variables below and rerun the subsequent cells. This creates temporary copies of the programs with matching matrix sizes and run counts, keeping the exercise sources unchanged.

Each program needs three double-precision matrices: approximately `3 × N × N × 8` bytes per memory space. The GPU program allocates both host and device copies. Start small before increasing the size.

In [4]:
from pathlib import Path
import re

matrix_size = 1000
runs = 5

assert type(matrix_size) is int and matrix_size > 0
assert type(runs) is int and runs > 0
output_dir = Path("performance-output")
output_dir.mkdir(exist_ok=True)
for filename in ["perf-gemm-cuda.cu", "perf-gemm-intelmkl.c"]:
    source = (Path("exercises/01-performance-basics") / filename).read_text()
    source, sizes_changed = re.subn(r"\bint N = \d+;", f"int N = {matrix_size};", source)
    source, runs_changed = re.subn(r"\bint runs = \d+;", f"int runs = {runs};", source)
    assert sizes_changed == runs_changed == 1
    (output_dir / filename).write_text(source)
print(f"Matrix: {matrix_size} × {matrix_size}; runs: {runs}")
print(f"Three matrices: {3 * matrix_size**2 * 8 / 1024**2:.1f} MiB per memory space")

Matrix: 1000 × 1000; runs: 5
Three matrices: 22.9 MiB per memory space


## Compile

These commands use the same optimization flags and libraries as the exercise CMake configuration. The loaded MKL environment must provide the include and library search paths.

In [ ]:
!cmake -S exercises -B exercises/build -DCMAKE_BUILD_TYPE=Release -DEXERCISES_EXTRA_SOURCES="performance-output/perf-gemm-cuda.cu;performance-output/perf-gemm-intelmkl.c"
!cmake --build exercises/build --parallel 4 --target ex-extra-perf-gemm-cuda ex-extra-perf-gemm-intelmkl


## Run the CPU benchmark

Record the printed milliseconds for each run.

In [ ]:
!./exercises/build/ex-extra-perf-gemm-intelmkl

## Run the GPU benchmark

Record the printed milliseconds for each run. The first run may differ from later runs; inspect the full set before summarizing it.

In [ ]:
!./exercises/build/ex-extra-perf-gemm-cuda

## Compare performance

The CPU program measures the MKL call using a host wall-clock timer. The GPU program uses CUDA events around the cuBLAS call. GPU memory allocation and the initial CPU-to-GPU transfers are **outside** its timed region, and the program does not copy results back for verification. These numbers compare library computation intervals rather than end-to-end application times.

Both programs start with `A = 1`, `B = 2`, and `C = 0`. Since `beta = 1`, they accumulate the product on each run; mathematically, every element of `C` should be `2 × N × completed_runs`. The supplied programs report timings but do not check these results.

Enter your observed times below. The cell uses medians, reports the CPU/GPU time ratio, and estimates throughput using approximately `2 × N³` floating-point operations per matrix multiplication.

In [8]:
from statistics import median

cpu_times_ms = []  # Enter the printed CPU times here.
gpu_times_ms = []  # Enter the printed GPU times here.

if cpu_times_ms and gpu_times_ms:
    assert len(cpu_times_ms) == len(gpu_times_ms) == runs
    assert all(t > 0 for t in cpu_times_ms + gpu_times_ms)
    cpu_ms, gpu_ms = median(cpu_times_ms), median(gpu_times_ms)
    operations = 2 * matrix_size**3
    print(f"Median CPU time: {cpu_ms:.3f} ms; GPU time: {gpu_ms:.3f} ms")
    print(f"CPU time / GPU time: {cpu_ms / gpu_ms:.2f}")
    print(f"CPU throughput: {operations / (cpu_ms * 1e6):.2f} GFLOP/s")
    print(f"GPU throughput: {operations / (gpu_ms * 1e6):.2f} GFLOP/s")
else:
    print("Enter both timing lists above after running the benchmarks.")

Enter both timing lists above after running the benchmarks.


## Explore

1. Try `matrix_size = 256`, `1000`, and `2048`, recompiling and rerunning each time. Reset the timing lists for each configuration.
2. Does the CPU/GPU time ratio change with problem size?
3. How would including GPU allocation and memory transfers change the comparison?
4. What would need to change to compare against multithreaded MKL on multiple CPU cores?

### Observations

Record the GPU model, CPU configuration, matrix size, precision, run times, and what each timer includes.